In [2]:
import os
import sys
from pathlib import Path
import torch
from transformers import BeitForImageClassification, BeitImageProcessor, BeitConfig
from huggingface_hub import snapshot_download, hf_hub_download
import json
from tqdm import tqdm
import time
from torchinfo import summary 

In [ ]:
# Download the BEIT model and processor


class ModelDownloader:
    def __init__(self, model_name, save_path):
        self.model_name = model_name
        self.save_path = Path(save_path)
        self.model_path = self.save_path / "beit-base-patch16-224"
        
    def create_directories(self):
        """Create necessary directories"""
        print(f"Creating directory structure at: {self.save_path}")
        self.save_path.mkdir(parents=True, exist_ok=True)
        self.model_path.mkdir(parents=True, exist_ok=True)
        print(f"✓ Directory created: {self.model_path}")
        
    def download_model_with_progress(self):
        """Download model with progress bar"""
        print(f"\n🚀 Starting download of {self.model_name}")
        print(f"📁 Saving to: {self.model_path}")
        
        try:
            # Download the entire repository with progress
            downloaded_path = snapshot_download(
                repo_id=self.model_name,
                cache_dir=str(self.model_path),
                local_dir=str(self.model_path),
                local_dir_use_symlinks=False,  # Create actual files, not symlinks
                resume_download=True,
                force_download=False  # Don't re-download if already exists
            )
            
            print(f"✓ Model downloaded successfully to: {downloaded_path}")
            return True
            
        except Exception as e:
            print(f"❌ Error downloading model: {str(e)}")
            return False
    
    def verify_download(self):
        """Verify that all necessary files are downloaded"""
        print(f"\n🔍 Verifying download...")
        
        required_files = [
            "config.json",
            "pytorch_model.bin",
            "preprocessor_config.json"
        ]
        
        missing_files = []
        existing_files = []
        
        for file in required_files:
            file_path = self.model_path / file
            if file_path.exists():
                existing_files.append(file)
                file_size = file_path.stat().st_size / (1024 * 1024)  # Size in MB
                print(f"  ✓ {file}: {file_size:.2f} MB")
            else:
                missing_files.append(file)
                print(f"  ❌ {file}: Missing")
        
        # Check for additional files
        all_files = list(self.model_path.glob("*"))
        additional_files = [f.name for f in all_files if f.name not in required_files and f.is_file()]
        
        if additional_files:
            print(f"\n📄 Additional files found:")
            for file in additional_files:
                file_path = self.model_path / file
                file_size = file_path.stat().st_size / (1024 * 1024)
                print(f"  • {file}: {file_size:.2f} MB")
        
        return len(missing_files) == 0, missing_files
    
    def load_and_test_model(self):
        """Load the model and processor to verify they work correctly"""
        print(f"\n🔧 Loading model for verification...")
        
        try:
            # Load processor
            print("Loading image processor...")
            processor = BeitImageProcessor.from_pretrained(str(self.model_path))
            print("✓ Image processor loaded successfully")
            
            # Load model
            print("Loading model...")
            model = BeitForImageClassification.from_pretrained(str(self.model_path))
            print("✓ Model loaded successfully")
            
            # Test model forward pass with dummy input
            print("Testing model with dummy input...")
            dummy_input = torch.randn(1, 3, 224, 224)  # Batch size 1, 3 channels, 224x224
            
            model.eval()
            with torch.no_grad():
                outputs = model(dummy_input)
                logits = outputs.logits
                print(f"✓ Model forward pass successful. Output shape: {logits.shape}")
            
            return True, model, processor
            
        except Exception as e:
            print(f"❌ Error loading model: {str(e)}")
            return False, None, None
    
    def get_model_summary(self, model, processor):
        """Generate a comprehensive model summary"""
        print(f"\n📊 MODEL SUMMARY")
        print("=" * 50)
        
        # Basic model info
        config = model.config
        print(f"Model Name: {self.model_name}")
        print(f"Architecture: {config.architectures[0] if hasattr(config, 'architectures') else 'BEIT'}")
        print(f"Model Type: {type(model).__name__}")
        
        # Model configuration
        print(f"\n🏗️  ARCHITECTURE:")
        print(f"  • Hidden Size: {config.hidden_size}")
        print(f"  • Number of Layers: {config.num_hidden_layers}")
        print(f"  • Number of Attention Heads: {config.num_attention_heads}")
        print(f"  • Intermediate Size: {config.intermediate_size}")
        print(f"  • Image Size: {config.image_size}")
        print(f"  • Patch Size: {config.patch_size}")
        print(f"  • Number of Channels: {config.num_channels}")
        
        # Classification info
        if hasattr(config, 'num_labels'):
            print(f"  • Number of Classes: {config.num_labels}")
        
        # Model parameters
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        
        print(f"\n📈 PARAMETERS:")
        print(f"  • Total Parameters: {total_params:,}")
        print(f"  • Trainable Parameters: {trainable_params:,}")
        print(f"  • Model Size: ~{total_params * 4 / (1024**2):.1f} MB")
        
        # Processor info
        print(f"\n🖼️  IMAGE PROCESSOR:")
        if hasattr(processor, 'size'):
            print(f"  • Input Size: {processor.size}")
        if hasattr(processor, 'do_normalize') and processor.do_normalize:
            print(f"  • Normalization: Mean={processor.image_mean}, Std={processor.image_std}")
        if hasattr(processor, 'do_resize'):
            print(f"  • Resize: {processor.do_resize}")
        if hasattr(processor, 'do_center_crop'):
            print(f"  • Center Crop: {processor.do_center_crop}")
        
        # File information
        print(f"\n💾 STORAGE INFO:")
        total_size = sum(f.stat().st_size for f in self.model_path.rglob('*') if f.is_file())
        print(f"  • Total Download Size: {total_size / (1024**2):.1f} MB")
        print(f"  • Storage Location: {self.model_path}")
        print(f"  • Number of Files: {len(list(self.model_path.rglob('*')))}")
        
        # Usage recommendations
        print(f"\n💡 FINE-TUNING RECOMMENDATIONS:")
        print(f"  • Recommended Learning Rate: 1e-5 to 5e-5")
        print(f"  • Batch Size: 8-32 (depending on GPU memory)")
        print(f"  • Warm-up Steps: 10% of total training steps")
        print(f"  • Weight Decay: 0.01")
        print(f"  • Best for: Image classification tasks")
        
        return {
            'total_params': total_params,
            'trainable_params': trainable_params,
            'model_size_mb': total_params * 4 / (1024**2),
            'storage_size_mb': total_size / (1024**2),
            'config': config.to_dict() if hasattr(config, 'to_dict') else str(config)
        }
    
    def save_summary_to_file(self, summary_data):
        """Save model summary to a JSON file"""
        summary_file = self.model_path / "model_summary.json"
        
        # Convert any non-serializable objects to strings
        clean_summary = {}
        for key, value in summary_data.items():
            if isinstance(value, (int, float, str, list, dict)):
                clean_summary[key] = value
            else:
                clean_summary[key] = str(value)
        
        with open(summary_file, 'w', encoding='utf-8') as f:
            json.dump(clean_summary, f, indent=2, ensure_ascii=False)
        
        print(f"📄 Summary saved to: {summary_file}")

def main():
    # Configuration
    MODEL_NAME = "microsoft/beit-base-patch16-224-pt22k-ft22k"
    SAVE_PATH = "/Volumes/KODAK/folder_02/eye_disease_prediction_model/model/raw_model"
    
    print("🤖 BEIT Model Download and Verification Tool")
    print("=" * 60)
    
    # Initialize downloader
    downloader = ModelDownloader(MODEL_NAME, SAVE_PATH)
    
    try:
        # Step 1: Create directories
        downloader.create_directories()
        
        # Step 2: Download model
        download_success = downloader.download_model_with_progress()
        if not download_success:
            print("❌ Download failed. Exiting...")
            return
        
        # Step 3: Verify download
        verification_success, missing_files = downloader.verify_download()
        if not verification_success:
            print(f"❌ Verification failed. Missing files: {missing_files}")
            return
        else:
            print("✓ All required files downloaded successfully!")
        
        # Step 4: Load and test model
        load_success, model, processor = downloader.load_and_test_model()
        if not load_success:
            print("❌ Model loading failed. Exiting...")
            return
        
        # Step 5: Generate summary
        summary_data = downloader.get_model_summary(model, processor)
        
        # Step 6: Save summary
        downloader.save_summary_to_file(summary_data)
        
        print(f"\n🎉 SUCCESS! Model is ready for fine-tuning!")
        print(f"📍 Model location: {downloader.model_path}")
        print(f"\n🚀 Next steps:")
        print(f"   1. Prepare your eye disease dataset")
        print(f"   2. Set up data preprocessing")
        print(f"   3. Configure training parameters")
        print(f"   4. Start fine-tuning!")
        
    except KeyboardInterrupt:
        print(f"\n\n⚠️  Download interrupted by user")
    except Exception as e:
        print(f"\n❌ Unexpected error: {str(e)}")
        print(f"Please check your internet connection and try again.")

if __name__ == "__main__":
    main()

🤖 BEIT Model Download and Verification Tool
Creating directory structure at: /Volumes/KODAK/folder_02/eye_disease_prediction_model/model/raw_model
✓ Directory created: /Volumes/KODAK/folder_02/eye_disease_prediction_model/model/raw_model/beit-base-patch16-224

🚀 Starting download of microsoft/beit-base-patch16-224-pt22k-ft22k
📁 Saving to: /Volumes/KODAK/folder_02/eye_disease_prediction_model/model/raw_model/beit-base-patch16-224


/Volumes/KODAK/folder_02/eye_disease_prediction_model/venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:980: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(
/Volumes/KODAK/folder_02/eye_disease_prediction_model/venv/lib/python3.12/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/276 [00:00<?, ?B/s]

...

flax_model.msgpack:   0%|          | 0.00/410M [00:00<?, ?B/s]

✓ Model downloaded successfully to: /Volumes/KODAK/folder_02/eye_disease_prediction_model/model/raw_model/beit-base-patch16-224

🔍 Verifying download...
  ✓ config.json: 1.59 MB
  ✓ pytorch_model.bin: 394.87 MB
  ✓ preprocessor_config.json: 0.00 MB

📄 Additional files found:
  • ._models--microsoft--beit-base-patch16-224-pt22k-ft22k: 0.00 MB
  • ._.cache: 0.00 MB
  • README.md: 0.01 MB
  • ._README.md: 0.00 MB
  • ._preprocessor_config.json: 0.00 MB
  • .gitattributes: 0.00 MB
  • ._.gitattributes: 0.00 MB
  • ._config.json: 0.00 MB
  • ._pytorch_model.bin: 0.00 MB
  • flax_model.msgpack: 391.23 MB
  • ._flax_model.msgpack: 0.00 MB
✓ All required files downloaded successfully!

🔧 Loading model for verification...
Loading image processor...
✓ Image processor loaded successfully
Loading model...


/Volumes/KODAK/folder_02/eye_disease_prediction_model/venv/lib/python3.12/site-packages/transformers/utils/deprecation.py:172: UserWarning: The following named arguments are not valid for `BeitImageProcessor.__init__` and were ignored: 'feature_extractor_type'
  return func(*args, **kwargs)


✓ Model loaded successfully
Testing model with dummy input...
✓ Model forward pass successful. Output shape: torch.Size([1, 21841])

📊 MODEL SUMMARY
Model Name: microsoft/beit-base-patch16-224-pt22k-ft22k
Architecture: BeitForImageClassification
Model Type: BeitForImageClassification

🏗️  ARCHITECTURE:
  • Hidden Size: 768
  • Number of Layers: 12
  • Number of Attention Heads: 12
  • Intermediate Size: 3072
  • Image Size: 224
  • Patch Size: 16
  • Number of Channels: 3
  • Number of Classes: 21841

📈 PARAMETERS:
  • Total Parameters: 102,557,713
  • Trainable Parameters: 102,557,713
  • Model Size: ~391.2 MB

🖼️  IMAGE PROCESSOR:
  • Input Size: {'height': 224, 'width': 224}
  • Normalization: Mean=[0.5, 0.5, 0.5], Std=[0.5, 0.5, 0.5]
  • Resize: True
  • Center Crop: False

💾 STORAGE INFO:
  • Total Download Size: 787.8 MB
  • Storage Location: /Volumes/KODAK/folder_02/eye_disease_prediction_model/model/raw_model/beit-base-patch16-224
  • Number of Files: 50

💡 FINE-TUNING RECOMMEN

: 

In [4]:
# Adjusted model loading script for eye disease prediction



# === Step 1: Define Paths ===
RAW_MODEL_PATH = "/Volumes/KODAK/folder_02/eye_disease_prediction_model/model/raw_model/beit-base-patch16-224"
OUTPUT_MODEL_DIR = "/Volumes/KODAK/folder_02/eye_disease_prediction_model/model/process_model/beit-adjusted"
os.makedirs(OUTPUT_MODEL_DIR, exist_ok=True)

# === Step 2: Define class labels ===
class_names = [
    "Age_related_Macular_Degeneration",
    "Cataract",
    "Diabetes",
    "Glaucoma",
    "Hypertension",
    "Normal",
    "Other_diseases",
    "Pathological_Myopia"
]
id2label = {i: label for i, label in enumerate(class_names)}
label2id = {label: i for i, label in enumerate(class_names)}

# === Step 3: Load base config and override classifier info ===
config = BeitConfig.from_pretrained(
    RAW_MODEL_PATH,
    num_labels=len(class_names),
    id2label=id2label,
    label2id=label2id,
)

# === Step 4: Load model without classifier ===
model = BeitForImageClassification.from_pretrained(
    RAW_MODEL_PATH,
    config=config,
    ignore_mismatched_sizes=True  # THIS is key to avoid size mismatch errors
)

# === Step 5: Switch to eval mode ===
model.eval()

# === Step 6: Load and Save Preprocessor ===
processor = BeitImageProcessor.from_pretrained(RAW_MODEL_PATH)
processor.save_pretrained(OUTPUT_MODEL_DIR)

# === Step 7: Save adjusted model ===
model.save_pretrained(OUTPUT_MODEL_DIR)

# === Step 8: Model Summary ===
print("\n✅ Model successfully adjusted and saved.")
print("\n🧠 Model Summary:")
summary(model, input_size=(1, 3, 224, 224), device="cpu")

# === Step 9: Optional - Show label mappings ===
print("\n🧪 Label Mapping (id2label):")
print(json.dumps(id2label, indent=2))


Some weights of BeitForImageClassification were not initialized from the model checkpoint at /Volumes/KODAK/folder_02/eye_disease_prediction_model/model/raw_model/beit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([21841, 768]) in the checkpoint and torch.Size([8, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([21841]) in the checkpoint and torch.Size([8]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/Volumes/KODAK/folder_02/eye_disease_prediction_model/venv/lib/python3.12/site-packages/transformers/utils/deprecation.py:172: UserWarning: The following named arguments are not valid for `BeitImageProcessor.__init__` and were ignored: 'feature_extractor_type'
  return func(*args, **kwargs)



✅ Model successfully adjusted and saved.

🧠 Model Summary:

🧪 Label Mapping (id2label):
{
  "0": "Age_related_Macular_Degeneration",
  "1": "Cataract",
  "2": "Diabetes",
  "3": "Glaucoma",
  "4": "Hypertension",
  "5": "Normal",
  "6": "Other_diseases",
  "7": "Pathological_Myopia"
}


In [5]:
# Code to Display Full Summary of Adjusted BEiT Model


# === Step 1: Path to adjusted model ===
MODEL_PATH = "/Volumes/KODAK/folder_02/eye_disease_prediction_model/model/process_model/beit-adjusted"

# === Step 2: Load the adjusted model ===
model = BeitForImageClassification.from_pretrained(MODEL_PATH)

# === Step 3: Switch to evaluation mode ===
model.eval()

# === Step 4: Move model to CPU (safe) ===
device = torch.device("cpu")
model.to(device)

# === Step 5: Input size for summary (BEiT expects 224x224 RGB images) ===
input_size = (1, 3, 224, 224)

# === Step 6: Print full model summary ===
summary(model, input_size=input_size, depth=4, col_names=["input_size", "output_size", "num_params", "trainable"])


Layer (type:depth-idx)                                                 Input Shape               Output Shape              Param #                   Trainable
BeitForImageClassification                                             [1, 3, 224, 224]          [1, 8]                    --                        True
├─BeitModel: 1-1                                                       [1, 3, 224, 224]          [1, 768]                  --                        True
│    └─BeitEmbeddings: 2-1                                             [1, 3, 224, 224]          [1, 197, 768]             768                       True
│    │    └─BeitPatchEmbeddings: 3-1                                   [1, 3, 224, 224]          [1, 196, 768]             --                        True
│    │    │    └─Conv2d: 4-1                                           [1, 3, 224, 224]          [1, 768, 14, 14]          590,592                   True
│    │    └─Dropout: 3-2                                               

In [3]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import logging
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from PIL import Image
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.metrics import precision_recall_fscore_support
from transformers import (
    BeitForImageClassification, 
    BeitImageProcessor,
    get_cosine_schedule_with_warmup,
    get_linear_schedule_with_warmup
)
from tqdm import tqdm
import gc

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('fine_tuning.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

class EyeDiseaseDataset(Dataset):
    """Custom dataset for eye disease classification with left/right eye handling"""
    
    def __init__(self, data_dir, split='train', transform=None, processor=None):
        self.data_dir = Path(data_dir) / split
        self.transform = transform
        self.processor = processor
        self.classes = [
            'Age_related_Macular_Degeneration', 'Cataract', 'Diabetes', 
            'Glaucoma', 'Hypertension', 'Normal', 'Other_diseases', 'Pathological_Myopia'
        ]
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        self.samples = self._load_samples()
        
    def _load_samples(self):
        samples = []
        for class_name in self.classes:
            class_dir = self.data_dir / class_name
            if not class_dir.exists():
                logger.warning(f"Class directory {class_dir} not found")
                continue
                
            for eye_type in ['left_eye', 'right_eye']:
                eye_dir = class_dir / eye_type
                if not eye_dir.exists():
                    continue
                    
                for img_path in eye_dir.glob('*'):
                    if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                        samples.append({
                            'path': img_path,
                            'label': self.class_to_idx[class_name],
                            'class_name': class_name,
                            'eye_type': eye_type
                        })
        
        logger.info(f"Loaded {len(samples)} samples for split: {self.data_dir.name}")
        return samples
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        try:
            image = Image.open(sample['path']).convert('RGB')
            
            if self.processor:
                # Use BEiT processor
                inputs = self.processor(images=image, return_tensors="pt")
                pixel_values = inputs['pixel_values'].squeeze(0)
            else:
                # Use torchvision transforms
                if self.transform:
                    image = self.transform(image)
                pixel_values = image
            
            return {
                'pixel_values': pixel_values,
                'labels': torch.tensor(sample['label'], dtype=torch.long),
                'class_name': sample['class_name'],
                'eye_type': sample['eye_type']
            }
        except Exception as e:
            logger.error(f"Error loading image {sample['path']}: {e}")
            # Return a dummy sample
            if self.processor:
                dummy_image = Image.new('RGB', (224, 224), color='black')
                inputs = self.processor(images=dummy_image, return_tensors="pt")
                pixel_values = inputs['pixel_values'].squeeze(0)
            else:
                pixel_values = torch.zeros(3, 224, 224)
            
            return {
                'pixel_values': pixel_values,
                'labels': torch.tensor(0, dtype=torch.long),
                'class_name': 'Normal',
                'eye_type': 'left_eye'
            }

class AdvancedTrainingMetrics:
    """Comprehensive training metrics tracking"""
    
    def __init__(self, num_classes, class_names):
        self.num_classes = num_classes
        self.class_names = class_names
        self.reset()
    
    def reset(self):
        self.train_losses = []
        self.train_accuracies = []
        self.val_losses = []
        self.val_accuracies = []
        self.learning_rates = []
        self.epoch_times = []
        self.per_class_metrics = []
        self.confusion_matrices = []
        
    def update_epoch(self, train_loss, train_acc, val_loss, val_acc, lr, epoch_time, 
                    per_class_acc, confusion_mat):
        self.train_losses.append(train_loss)
        self.train_accuracies.append(train_acc)
        self.val_losses.append(val_loss)
        self.val_accuracies.append(val_acc)
        self.learning_rates.append(lr)
        self.epoch_times.append(epoch_time)
        self.per_class_metrics.append(per_class_acc)
        self.confusion_matrices.append(confusion_mat)
    
    def get_summary_stats(self):
        return {
            'best_val_acc': max(self.val_accuracies) if self.val_accuracies else 0,
            'best_val_acc_epoch': np.argmax(self.val_accuracies) + 1 if self.val_accuracies else 0,
            'final_train_acc': self.train_accuracies[-1] if self.train_accuracies else 0,
            'final_val_acc': self.val_accuracies[-1] if self.val_accuracies else 0,
            'total_training_time': sum(self.epoch_times),
            'avg_epoch_time': np.mean(self.epoch_times) if self.epoch_times else 0
        }

class EyeDiseaseFinetuner:
    """Industrial-level fine-tuning pipeline for eye disease classification"""
    
    def __init__(self, model_path, data_path, output_path, config=None):
        self.model_path = Path(model_path)
        self.data_path = Path(data_path)
        self.output_path = Path(output_path)
        self.output_path.mkdir(parents=True, exist_ok=True)
        
        # Default configuration
        self.config = {
            'batch_size': 16,
            'num_epochs': 50,
            'learning_rate': 2e-5,
            'weight_decay': 0.01,
            'warmup_ratio': 0.1,
            'patience': 10,
            'min_delta': 0.001,
            'freeze_epochs': 10,  # Number of epochs to keep backbone frozen
            'grad_clip_norm': 1.0,
            'label_smoothing': 0.1,
            'scheduler_type': 'cosine',  # 'cosine' or 'linear'
            'accumulation_steps': 1,
            'mixed_precision': True
        }
        
        if config:
            self.config.update(config)
        
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        logger.info(f"Using device: {self.device}")
        
        # Initialize components
        self.model = None
        self.processor = None
        self.class_names = [
            'Age_related_Macular_Degeneration', 'Cataract', 'Diabetes', 
            'Glaucoma', 'Hypertension', 'Normal', 'Other_diseases', 'Pathological_Myopia'
        ]
        self.num_classes = len(self.class_names)
        self.metrics = AdvancedTrainingMetrics(self.num_classes, self.class_names)
        
    def load_model_and_processor(self):
        """Load pre-trained model and processor"""
        try:
            logger.info(f"Loading model from {self.model_path}")
            
            # Load processor
            self.processor = BeitImageProcessor.from_pretrained(self.model_path)
            
            # Load model
            self.model = BeitForImageClassification.from_pretrained(
                self.model_path,
                num_labels=self.num_classes,
                ignore_mismatched_sizes=True
            )
            
            # Modify classifier head if needed
            if self.model.classifier.out_features != self.num_classes:
                self.model.classifier = nn.Linear(
                    self.model.classifier.in_features, 
                    self.num_classes
                )
                logger.info(f"Modified classifier head to {self.num_classes} classes")
            
            self.model.to(self.device)
            logger.info("Model and processor loaded successfully")
            
        except Exception as e:
            logger.error(f"Error loading model: {e}")
            raise
    
    def create_data_loaders(self):
        """Create data loaders with proper augmentation"""
        # Data augmentation for training
        train_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        # Validation/Test transforms
        val_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])
        
        # Create datasets
        train_dataset = EyeDiseaseDataset(
            self.data_path, 'train', processor=self.processor
        )
        val_dataset = EyeDiseaseDataset(
            self.data_path, 'val', processor=self.processor
        )
        test_dataset = EyeDiseaseDataset(
            self.data_path, 'test', processor=self.processor
        )
        
        # Create data loaders with num_workers=0 to avoid multiprocessing issues
        self.train_loader = DataLoader(
            train_dataset, 
            batch_size=self.config['batch_size'],
            shuffle=True,
            num_workers=0,  # Fixed: Set to 0 to avoid multiprocessing issues
            pin_memory=False,  # Fixed: Set to False when using CPU
            drop_last=True
        )
        
        self.val_loader = DataLoader(
            val_dataset,
            batch_size=self.config['batch_size'],
            shuffle=False,
            num_workers=0,  # Fixed: Set to 0 to avoid multiprocessing issues
            pin_memory=False  # Fixed: Set to False when using CPU
        )
        
        self.test_loader = DataLoader(
            test_dataset,
            batch_size=self.config['batch_size'],
            shuffle=False,
            num_workers=0,  # Fixed: Set to 0 to avoid multiprocessing issues
            pin_memory=False  # Fixed: Set to False when using CPU
        )
        
        logger.info(f"Created data loaders - Train: {len(train_dataset)}, "
                   f"Val: {len(val_dataset)}, Test: {len(test_dataset)}")
        
        return train_dataset, val_dataset, test_dataset
    
    def setup_training_components(self, total_steps):
        """Setup optimizer, scheduler, and loss function"""
        # Freeze/unfreeze strategy
        if self.config['freeze_epochs'] > 0:
            self.freeze_backbone()
        
        # Optimizer with different learning rates for different parts
        backbone_params = []
        classifier_params = []
        
        for name, param in self.model.named_parameters():
            if 'classifier' in name:
                classifier_params.append(param)
            else:
                backbone_params.append(param)
        
        optimizer_params = [
            {'params': backbone_params, 'lr': self.config['learning_rate'] * 0.1},
            {'params': classifier_params, 'lr': self.config['learning_rate']}
        ]
        
        self.optimizer = optim.AdamW(
            optimizer_params,
            weight_decay=self.config['weight_decay'],
            eps=1e-8
        )
        
        # Scheduler
        warmup_steps = int(total_steps * self.config['warmup_ratio'])
        
        if self.config['scheduler_type'] == 'cosine':
            self.scheduler = get_cosine_schedule_with_warmup(
                self.optimizer,
                num_warmup_steps=warmup_steps,
                num_training_steps=total_steps
            )
        else:
            self.scheduler = get_linear_schedule_with_warmup(
                self.optimizer,
                num_warmup_steps=warmup_steps,
                num_training_steps=total_steps
            )
        
        # Loss function with label smoothing
        self.criterion = nn.CrossEntropyLoss(
            label_smoothing=self.config['label_smoothing']
        )
        
        # Mixed precision scaler - only if using CUDA
        self.scaler = torch.cuda.amp.GradScaler() if (self.config['mixed_precision'] and torch.cuda.is_available()) else None
        
        logger.info("Training components setup complete")
    
    def freeze_backbone(self):
        """Freeze backbone parameters"""
        for name, param in self.model.named_parameters():
            if 'classifier' not in name:
                param.requires_grad = False
        logger.info("Backbone frozen - only classifier will be trained")
    
    def unfreeze_backbone(self):
        """Unfreeze backbone parameters"""
        for param in self.model.parameters():
            param.requires_grad = True
        logger.info("Backbone unfrozen - full model will be trained")
    
    def train_epoch(self, epoch):
        """Train for one epoch"""
        self.model.train()
        total_loss = 0
        correct = 0
        total = 0
        
        # Progress bar
        pbar = tqdm(self.train_loader, desc=f'Epoch {epoch+1}/{self.config["num_epochs"]}')
        
        for batch_idx, batch in enumerate(pbar):
            pixel_values = batch['pixel_values'].to(self.device)
            labels = batch['labels'].to(self.device)
            
            # Forward pass with mixed precision (only if using CUDA)
            if self.scaler and torch.cuda.is_available():
                with torch.cuda.amp.autocast():
                    outputs = self.model(pixel_values=pixel_values)
                    loss = self.criterion(outputs.logits, labels)
                    loss = loss / self.config['accumulation_steps']
            else:
                outputs = self.model(pixel_values=pixel_values)
                loss = self.criterion(outputs.logits, labels)
                loss = loss / self.config['accumulation_steps']
            
            # Backward pass
            if self.scaler and torch.cuda.is_available():
                self.scaler.scale(loss).backward()
            else:
                loss.backward()
            
            # Gradient accumulation
            if (batch_idx + 1) % self.config['accumulation_steps'] == 0:
                if self.scaler and torch.cuda.is_available():
                    # Gradient clipping
                    self.scaler.unscale_(self.optimizer)
                    torch.nn.utils.clip_grad_norm_(
                        self.model.parameters(), 
                        self.config['grad_clip_norm']
                    )
                    
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(
                        self.model.parameters(), 
                        self.config['grad_clip_norm']
                    )
                    self.optimizer.step()
                
                self.scheduler.step()
                self.optimizer.zero_grad()
            
            # Statistics
            total_loss += loss.item() * self.config['accumulation_steps']
            _, predicted = outputs.logits.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            # Update progress bar
            current_lr = self.scheduler.get_last_lr()[0]
            pbar.set_postfix({
                'Loss': f'{loss.item():.4f}',
                'Acc': f'{100.*correct/total:.2f}%',
                'LR': f'{current_lr:.2e}'
            })
        
        epoch_loss = total_loss / len(self.train_loader)
        epoch_acc = 100. * correct / total
        
        return epoch_loss, epoch_acc, current_lr
    
    def validate_epoch(self):
        """Validate the model"""
        self.model.eval()
        total_loss = 0
        correct = 0
        total = 0
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for batch in tqdm(self.val_loader, desc='Validating'):
                pixel_values = batch['pixel_values'].to(self.device)
                labels = batch['labels'].to(self.device)
                
                if self.scaler and torch.cuda.is_available():
                    with torch.cuda.amp.autocast():
                        outputs = self.model(pixel_values=pixel_values)
                        loss = self.criterion(outputs.logits, labels)
                else:
                    outputs = self.model(pixel_values=pixel_values)
                    loss = self.criterion(outputs.logits, labels)
                
                total_loss += loss.item()
                _, predicted = outputs.logits.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
                
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
        
        epoch_loss = total_loss / len(self.val_loader)
        epoch_acc = 100. * correct / total
        
        # Per-class accuracy
        per_class_acc = {}
        cm = confusion_matrix(all_labels, all_preds)
        
        for i, class_name in enumerate(self.class_names):
            if cm[i, :].sum() > 0:
                per_class_acc[class_name] = cm[i, i] / cm[i, :].sum()
            else:
                per_class_acc[class_name] = 0.0
        
        return epoch_loss, epoch_acc, per_class_acc, cm
    
    def train(self):
        """Main training loop"""
        logger.info("Starting fine-tuning process...")
        
        # Load model and create data loaders
        self.load_model_and_processor()
        train_dataset, val_dataset, test_dataset = self.create_data_loaders()
        
        # Setup training components
        total_steps = len(self.train_loader) * self.config['num_epochs'] // self.config['accumulation_steps']
        self.setup_training_components(total_steps)
        
        # Training variables
        best_val_acc = 0
        patience_counter = 0
        start_time = datetime.now()
        
        # Save initial configuration
        config_path = self.output_path / 'training_config.json'
        with open(config_path, 'w') as f:
            json.dump(self.config, f, indent=2)
        
        for epoch in range(self.config['num_epochs']):
            epoch_start_time = datetime.now()
            
            # Unfreeze backbone after specified epochs
            if epoch == self.config['freeze_epochs'] and self.config['freeze_epochs'] > 0:
                self.unfreeze_backbone()
                # Reset optimizer with new parameters
                total_remaining_steps = (self.config['num_epochs'] - epoch) * len(self.train_loader) // self.config['accumulation_steps']
                self.setup_training_components(total_remaining_steps)
            
            # Training
            train_loss, train_acc, current_lr = self.train_epoch(epoch)
            
            # Validation
            val_loss, val_acc, per_class_acc, cm = self.validate_epoch()
            
            # Calculate epoch time
            epoch_time = (datetime.now() - epoch_start_time).total_seconds()
            
            # Update metrics
            self.metrics.update_epoch(
                train_loss, train_acc, val_loss, val_acc, 
                current_lr, epoch_time, per_class_acc, cm
            )
            
            # Logging
            logger.info(
                f"Epoch {epoch+1}/{self.config['num_epochs']} - "
                f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, "
                f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%, "
                f"LR: {current_lr:.2e}, Time: {epoch_time:.2f}s"
            )
            
            # Per-class accuracy logging
            for class_name, acc in per_class_acc.items():
                logger.info(f"  {class_name}: {acc:.3f}")
            
            # Save best model
            if val_acc > best_val_acc + self.config['min_delta']:
                best_val_acc = val_acc
                patience_counter = 0
                
                # Save best model
                best_model_path = self.output_path / f'best_model_epoch_{epoch+1}'
                self.model.save_pretrained(best_model_path)
                self.processor.save_pretrained(best_model_path)
                
                logger.info(f"New best model saved with validation accuracy: {val_acc:.2f}%")
            else:
                patience_counter += 1
            
            # Early stopping
            if patience_counter >= self.config['patience']:
                logger.info(f"Early stopping triggered after {epoch+1} epochs")
                break
            
            # Memory cleanup
            if epoch % 5 == 0:
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                gc.collect()
        
        # Training completed
        total_time = (datetime.now() - start_time).total_seconds()
        logger.info(f"Training completed in {total_time:.2f} seconds")
        
        # Save final model
        final_model_path = self.output_path / 'final_model'
        self.model.save_pretrained(final_model_path)
        self.processor.save_pretrained(final_model_path)
        
        # Generate comprehensive report
        self.generate_training_report()
        self.create_visualizations()
        
        return self.metrics.get_summary_stats()
    
    def test_model(self):
        """Evaluate model on test set"""
        logger.info("Evaluating on test set...")
        
        self.model.eval()
        all_preds = []
        all_labels = []
        all_probs = []
        
        with torch.no_grad():
            for batch in tqdm(self.test_loader, desc='Testing'):
                pixel_values = batch['pixel_values'].to(self.device)
                labels = batch['labels'].to(self.device)
                
                outputs = self.model(pixel_values=pixel_values)
                probs = F.softmax(outputs.logits, dim=1)
                _, predicted = outputs.logits.max(1)
                
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())
        
        # Calculate metrics
        test_acc = accuracy_score(all_labels, all_preds)
        precision, recall, f1, support = precision_recall_fscore_support(
            all_labels, all_preds, average='weighted'
        )
        
        # Classification report
        report = classification_report(
            all_labels, all_preds, 
            target_names=self.class_names,
            output_dict=True
        )
        
        # Confusion matrix
        cm = confusion_matrix(all_labels, all_preds)
        
        logger.info(f"Test Accuracy: {test_acc:.4f}")
        logger.info(f"Test Precision: {precision:.4f}")
        logger.info(f"Test Recall: {recall:.4f}")
        logger.info(f"Test F1-Score: {f1:.4f}")
        
        return {
            'accuracy': test_acc,
            'precision': precision,
            'recall': recall,
            'f1_score': f1,
            'classification_report': report,
            'confusion_matrix': cm,
            'predictions': all_preds,
            'labels': all_labels,
            'probabilities': all_probs
        }
    
    def create_visualizations(self):
        """Create comprehensive training visualizations"""
        plt.style.use('seaborn-v0_8')
        fig = plt.figure(figsize=(20, 15))
        
        # 1. Training and Validation Loss
        plt.subplot(2, 3, 1)
        epochs = range(1, len(self.metrics.train_losses) + 1)
        plt.plot(epochs, self.metrics.train_losses, 'b-', label='Training Loss', linewidth=2)
        plt.plot(epochs, self.metrics.val_losses, 'r-', label='Validation Loss', linewidth=2)
        plt.title('Training and Validation Loss', fontsize=14, fontweight='bold')
        plt.xlabel('Epoch')
        plt.ylabel('Loss')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # 2. Training and Validation Accuracy
        plt.subplot(2, 3, 2)
        plt.plot(epochs, self.metrics.train_accuracies, 'b-', label='Training Accuracy', linewidth=2)
        plt.plot(epochs, self.metrics.val_accuracies, 'r-', label='Validation Accuracy', linewidth=2)
        plt.title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy (%)')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # 3. Learning Rate Schedule
        plt.subplot(2, 3, 3)
        plt.plot(epochs, self.metrics.learning_rates, 'g-', linewidth=2)
        plt.title('Learning Rate Schedule', fontsize=14, fontweight='bold')
        plt.xlabel('Epoch')
        plt.ylabel('Learning Rate')
        plt.yscale('log')
        plt.grid(True, alpha=0.3)
        
        # 4. Per-Class Accuracy Evolution
        plt.subplot(2, 3, 4)
        if self.metrics.per_class_metrics:
            for class_name in self.class_names:
                class_accs = [metrics.get(class_name, 0) for metrics in self.metrics.per_class_metrics]
                plt.plot(epochs, class_accs, label=class_name, linewidth=1.5)
        plt.title('Per-Class Accuracy Evolution', fontsize=14, fontweight='bold')
        plt.xlabel('Epoch')
        plt.ylabel('Accuracy')
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.grid(True, alpha=0.3)
        
        # 5. Training Time per Epoch
        plt.subplot(2, 3, 5)
        plt.plot(epochs, self.metrics.epoch_times, 'purple', linewidth=2)
        plt.title('Training Time per Epoch', fontsize=14, fontweight='bold')
        plt.xlabel('Epoch')
        plt.ylabel('Time (seconds)')
        plt.grid(True, alpha=0.3)
        
        # 6. Final Confusion Matrix
        plt.subplot(2, 3, 6)
        if self.metrics.confusion_matrices:
            final_cm = self.metrics.confusion_matrices[-1]
            sns.heatmap(final_cm, annot=True, fmt='d', cmap='Blues',
                       xticklabels=[name[:10] for name in self.class_names],
                       yticklabels=[name[:10] for name in self.class_names])
            plt.title('Final Confusion Matrix', fontsize=14, fontweight='bold')
            plt.xlabel('Predicted')
            plt.ylabel('Actual')
        
        plt.tight_layout()
        plt.savefig(self.output_path / 'training_visualization.png', dpi=300, bbox_inches='tight')
        plt.close()
        
        # Create separate detailed confusion matrix heatmap
        self.create_detailed_confusion_matrix()
        
        logger.info("Training visualizations saved")
    
    def create_detailed_confusion_matrix(self):
        """Create a detailed confusion matrix heatmap"""
        if not self.metrics.confusion_matrices:
            return
        
        plt.figure(figsize=(12, 10))
        final_cm = self.metrics.confusion_matrices[-1]
        
        # Normalize confusion matrix
        cm_normalized = final_cm.astype('float') / final_cm.sum(axis=1)[:, np.newaxis]
        
        # Create heatmap
        sns.heatmap(cm_normalized, annot=True, fmt='.3f', cmap='Blues',
                   xticklabels=self.class_names, yticklabels=self.class_names,
                   cbar_kws={'label': 'Normalized Count'})
        
        plt.title('Detailed Confusion Matrix (Normalized)', fontsize=16, fontweight='bold')
        plt.xlabel('Predicted Class', fontsize=12)
        plt.ylabel('True Class', fontsize=12)
        plt.xticks(rotation=45, ha='right')
        plt.yticks(rotation=0)
        
        plt.tight_layout()
        plt.savefig(self.output_path / 'detailed_confusion_matrix.png', dpi=300, bbox_inches='tight')
        plt.close()
        
        logger.info("Detailed confusion matrix saved")
    
    def generate_training_report(self):
        """Generate comprehensive training report"""
        stats = self.metrics.get_summary_stats()
        
        report = {
            'training_summary': {
                'total_epochs': len(self.metrics.train_losses),
                'best_validation_accuracy': stats['best_val_acc'],
                'best_epoch': stats['best_val_acc_epoch'],
                'final_training_accuracy': stats['final_train_acc'],
                'final_validation_accuracy': stats['final_val_acc'],
                'total_training_time_hours': stats['total_training_time'] / 3600,
                'average_epoch_time_minutes': stats['avg_epoch_time'] / 60
            },
            'configuration': self.config,
            'model_info': {
                'model_path': str(self.model_path),
                'num_classes': self.num_classes,
                'class_names': self.class_names
            },
            'training_curves': {
                'train_losses': self.metrics.train_losses,
                'train_accuracies': self.metrics.train_accuracies,
                'val_losses': self.metrics.val_losses,
                'val_accuracies': self.metrics.val_accuracies,
                'learning_rates': self.metrics.learning_rates
            }
        }
        
        # Save report
        report_path = self.output_path / 'training_report.json'
        with open(report_path, 'w') as f:
            json.dump(report, f, indent=2)
        
        # Create detailed text report
        text_report_path = self.output_path / 'training_report.txt'
        with open(text_report_path, 'w') as f:
            f.write("="*80 + "\n")
            f.write("EYE DISEASE CLASSIFICATION - FINE-TUNING REPORT\n")
            f.write("="*80 + "\n\n")
            
            f.write("TRAINING SUMMARY:\n")
            f.write("-"*40 + "\n")
            f.write(f"Total Epochs: {report['training_summary']['total_epochs']}\n")
            f.write(f"Best Validation Accuracy: {report['training_summary']['best_validation_accuracy']:.2f}%\n")
            f.write(f"Best Epoch: {report['training_summary']['best_epoch']}\n")
            f.write(f"Final Training Accuracy: {report['training_summary']['final_training_accuracy']:.2f}%\n")
            f.write(f"Final Validation Accuracy: {report['training_summary']['final_validation_accuracy']:.2f}%\n")
            f.write(f"Total Training Time: {report['training_summary']['total_training_time_hours']:.2f} hours\n")
            f.write(f"Average Epoch Time: {report['training_summary']['average_epoch_time_minutes']:.2f} minutes\n\n")
            
            f.write("CONFIGURATION:\n")
            f.write("-"*40 + "\n")
            for key, value in self.config.items():
                f.write(f"{key}: {value}\n")
            f.write("\n")
            
            f.write("CLASSES:\n")
            f.write("-"*40 + "\n")
            for i, class_name in enumerate(self.class_names):
                f.write(f"{i}: {class_name}\n")
        
        logger.info("Training report generated")


def main():
    """Main execution function"""
    
    # Configuration
    config = {
        'batch_size': 8,  # Reduced batch size for CPU training
        'num_epochs': 50,  # Reduced epochs for initial testing
        'learning_rate': 2e-5,
        'weight_decay': 0.01,
        'warmup_ratio': 0.1,
        'patience': 15,
        'min_delta': 0.001,
        'freeze_epochs': 10,  # Freeze backbone for first 10 epochs
        'grad_clip_norm': 1.0,
        'label_smoothing': 0.1,
        'scheduler_type': 'cosine',
        'accumulation_steps': 4,  # Effective batch size = 8 * 4 = 32
        'mixed_precision': False  # Disabled for CPU training
    }
    
    # Paths
    model_path = '/Volumes/KODAK/folder_02/eye_disease_prediction_model/model/process_model/beit-adjusted'
    data_path = '/Volumes/KODAK/folder_02/eye_disease_prediction_model/data/preprocess_data/splitted_data'
    output_path = '/Volumes/KODAK/folder_02/eye_disease_prediction_model/model/fine_tune_model'
    
    # Create output directory with timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    final_output_path = Path(output_path) / f'fine_tuned_model_{timestamp}'
    
    try:
        # Initialize fine-tuner
        finetuner = EyeDiseaseFinetuner(
            model_path=model_path,
            data_path=data_path,
            output_path=final_output_path,
            config=config
        )
        
        # Start training
        logger.info("Starting industrial-level fine-tuning pipeline...")
        training_stats = finetuner.train()
        
        # Test the final model
        test_results = finetuner.test_model()
        
        # Print final results
        print("\n" + "="*80)
        print("FINE-TUNING COMPLETED SUCCESSFULLY!")
        print("="*80)
        print(f"Best Validation Accuracy: {training_stats['best_val_acc']:.2f}%")
        print(f"Final Test Accuracy: {test_results['accuracy']*100:.2f}%")
        print(f"Test F1-Score: {test_results['f1_score']:.4f}")
        print(f"Total Training Time: {training_stats['total_training_time']/3600:.2f} hours")
        print(f"Model saved to: {final_output_path}")
        
        # Save test results
        test_report_path = final_output_path / 'test_results.json'
        with open(test_report_path, 'w') as f:
            # Convert numpy arrays to lists for JSON serialization
            test_results_serializable = {
                'accuracy': float(test_results['accuracy']),
                'precision': float(test_results['precision']),
                'recall': float(test_results['recall']),
                'f1_score': float(test_results['f1_score']),
                'classification_report': test_results['classification_report'],
                'confusion_matrix': test_results['confusion_matrix'].tolist(),
                'predictions': test_results['predictions'],
                'labels': test_results['labels']
            }
            json.dump(test_results_serializable, f, indent=2)
        
        logger.info("All outputs saved successfully!")
        
        if test_results['accuracy'] > 0.95:
            print("\n🎉 CONGRATULATIONS! Achieved >95% accuracy target!")
        else:
            print(f"\n📊 Current accuracy: {test_results['accuracy']*100:.2f}%")
            print("💡 Suggestions to reach 95%:")
            print("- Increase number of epochs")
            print("- Adjust learning rate schedule")
            print("- Try different data augmentation strategies")
            print("- Consider ensemble methods")
            print("- Use GPU for faster training with larger batch sizes")
        
    except Exception as e:
        logger.error(f"Training failed: {e}")
        raise

if __name__ == "__main__":
    main()

2025-08-01 15:14:27,190 - INFO - Using device: cpu
2025-08-01 15:14:27,192 - INFO - Starting industrial-level fine-tuning pipeline...
2025-08-01 15:14:27,193 - INFO - Starting fine-tuning process...
2025-08-01 15:14:27,193 - INFO - Loading model from /Volumes/KODAK/folder_02/eye_disease_prediction_model/model/process_model/beit-adjusted
2025-08-01 15:14:27,285 - INFO - Model and processor loaded successfully
2025-08-01 15:14:27,421 - INFO - Loaded 11458 samples for split: train
2025-08-01 15:14:27,747 - INFO - Loaded 2454 samples for split: val
2025-08-01 15:14:27,781 - INFO - Loaded 2458 samples for split: test
2025-08-01 15:14:27,786 - INFO - Created data loaders - Train: 11458, Val: 2454, Test: 2458
2025-08-01 15:14:27,808 - INFO - Backbone frozen - only classifier will be trained
2025-08-01 15:14:27,809 - INFO - Training components setup complete
Validating: 100%|██████████| 307/307 [01:59<00:00,  2.57it/s]
2025-08-01 15:26:01,039 - INFO - Epoch 1/50 - Train Loss: 2.2045, Train Acc

KeyboardInterrupt: 

# Fine-Tuning Script

In [4]:
import os
import torch
import shutil
import random
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import BeitForImageClassification, BeitImageProcessor, TrainingArguments, Trainer
from torchvision import transforms
from torch.utils.data import Dataset
from sklearn.metrics import classification_report, confusion_matrix
from PIL import Image

# Paths
model_path = '/Volumes/KODAK/folder_02/eye_disease_prediction_model/model/process_model/beit-adjusted'
save_path = '/Volumes/KODAK/folder_02/eye_disease_prediction_model/model/fine_tune_model'
data_path = '/Volumes/KODAK/folder_02/eye_disease_prediction_model/data/preprocess_data/splitted_data'

# Constants
EPOCHS = 25
BATCH_SIZE = 8
IMG_SIZE = 224
SEED = 42
torch.manual_seed(SEED)

# Classes
labels = ['Age_related_Macular_Degeneration', 'Cataract', 'Diabetes', 'Glaucoma', 'Hypertension', 'Normal', 'Other_diseases', 'Pathological_Myopia']
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}


# Dataset Loader

In [5]:
class EyeDiseaseDataset(Dataset):
    def __init__(self, root_dir, processor, transform=None):
        self.processor = processor
        self.transform = transform
        self.samples = []

        for label in os.listdir(root_dir):
            for eye_type in ['left_eye', 'right_eye']:
                folder_path = os.path.join(root_dir, label, eye_type)
                if not os.path.isdir(folder_path): continue
                for file in os.listdir(folder_path):
                    if file.startswith("._") or not file.lower().endswith(('.jpg', '.jpeg', '.png')):
                        continue
                    self.samples.append((os.path.join(folder_path, file), label2id[label]))

        random.shuffle(self.samples)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        image_path, label = self.samples[idx]
        image = Image.open(image_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        inputs = self.processor(images=image, return_tensors="pt")
        inputs = {k: v.squeeze(0) for k, v in inputs.items()}
        inputs["labels"] = torch.tensor(label)
        return inputs


# Transforms & Dataset Init

In [6]:
processor = BeitImageProcessor.from_pretrained(model_path)
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])

train_dataset = EyeDiseaseDataset(os.path.join(data_path, 'train'), processor, transform)
val_dataset = EyeDiseaseDataset(os.path.join(data_path, 'val'), processor, transform)
test_dataset = EyeDiseaseDataset(os.path.join(data_path, 'test'), processor, transform)


# Model & Gradual Unfreezing Strategy

In [7]:
model = BeitForImageClassification.from_pretrained(
    model_path,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
)

# Freeze backbone initially
for param in model.beit.parameters():
    param.requires_grad = False

# Gradually unfreeze last few layers
for name, param in list(model.beit.named_parameters())[-24:]:  # Last N layers
    param.requires_grad = True


# Metrics & Callbacks

In [8]:
from sklearn.metrics import accuracy_score

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc}


# TrainingArguments & Trainer

In [9]:
args = TrainingArguments(
    output_dir=save_path,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    logging_dir=os.path.join(save_path, "logs"),
    num_train_epochs=EPOCHS,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_steps=10,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=processor,
    compute_metrics=compute_metrics
)

trainer.train()
trainer.save_model(os.path.join(save_path, "final_model"))


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Currently logged in as: malindap288 (malindap288-nsbm) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


It looks like you are trying to rescale already rescaled images. If the input images have pixel values between 0 and 1, set `do_rescale=False` to avoid rescaling them again.


Epoch,Training Loss,Validation Loss,Accuracy
1,1.785200,1.732414,0.345151
2,1.804200,1.617019,0.394051
3,1.572900,1.551389,0.427058
4,1.432100,1.516499,0.435208
5,1.766200,1.484869,0.442135
6,1.501500,1.435070,0.471068
7,1.655900,1.421285,0.470660
8,1.348200,1.371310,0.498778
9,1.510000,1.356755,0.495518
10,1.468600,1.344206,0.501222


KeyboardInterrupt: 

Error in callback <bound method _WandbInit._post_run_cell_hook of <wandb.sdk.wandb_init._WandbInit object at 0x1767da2a0>> (for post_run_cell), with arguments args (<ExecutionResult object at 14e53bc20, execution_count=9 error_before_exec=None error_in_exec= info=<ExecutionInfo object at 14e53ade0, raw_cell="args = TrainingArguments(
    output_dir=save_path.." transformed_cell="args = TrainingArguments(
    output_dir=save_path.." store_history=True silent=False shell_futures=True cell_id=vscode-notebook-cell:/Volumes/KODAK/folder_02/eye_disease_prediction_model/utils/model_loader.ipynb#X13sZmlsZQ%3D%3D> result=None>,),kwargs {}:


BrokenPipeError: [Errno 32] Broken pipe

# Evaluation, Plots & Heatmap

In [ ]:
import numpy as np

# Get predictions
preds = trainer.predict(test_dataset)
y_true = preds.label_ids
y_pred = np.argmax(preds.predictions, axis=1)

# Classification report
print(classification_report(y_true, y_pred, target_names=labels))

# Confusion matrix heatmap
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='Blues')
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(os.path.join(save_path, "confusion_matrix.png"))
plt.close()


# Plot Accuracy & Loss Curves

In [ ]:
training_log = trainer.state.log_history

train_acc = [x['eval_accuracy'] for x in training_log if 'eval_accuracy' in x]
train_loss = [x['loss'] for x in training_log if 'loss' in x]
epochs = list(range(1, len(train_acc)+1))

# Accuracy
plt.plot(epochs, train_acc, marker='o', label='Validation Accuracy')
plt.title('Validation Accuracy Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.grid()
plt.legend()
plt.savefig(os.path.join(save_path, "accuracy_curve.png"))
plt.close()

# Loss
plt.plot(range(1, len(train_loss)+1), train_loss, marker='x', label='Training Loss')
plt.title('Training Loss Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid()
plt.legend()
plt.savefig(os.path.join(save_path, "loss_curve.png"))
plt.close()
